# Week 2 practice: the Tennessee STAR experiment

In this notebook you repeat **Lab 2** (analyse an experiment) and **Lab 3** (see what goes wrong without randomization) on a new data set, mostly on your own.

**How to work.**

- Cells marked `# TODO` are yours. Replace each `None` with your code.
- After each `# TODO` cell there is a **check** cell. It prints `not done yet` until you fill in the cell above, and then tells you whether your answer matches.
- Write your answers to the questions in the empty text cells.
- Every cell must still run from top to bottom when you push: the automatic check on GitHub runs this notebook too.

## About the experiment

Project STAR (Student/Teacher Achievement Ratio) was run in Tennessee from 1985. Within each participating school, kindergarten students and teachers were **randomly assigned** to one of three class types (Krueger, 1999):

- **small**: about 13 to 17 students,
- **regular**: about 22 to 25 students,
- **regular with aide**: a regular class with a full-time teacher's aide.

The question: **do smaller classes raise test scores?**

We use the version of the data in the R package AER (Stock and Watson, 2007), loaded through `statsmodels`. One row is one student. The columns we use:

| Column | Meaning |
|---|---|
| `stark` | class type in kindergarten: `small`, `regular`, `regular+aide`, or missing (student joined STAR later) |
| `readk` | reading score at the end of kindergarten (Stanford Achievement Test, scaled score) |
| `mathk` | math score at the end of kindergarten |
| `gender` | `male` or `female` |
| `ethnicity` | `cauc`, `afam`, and a few others |
| `lunchk` | `free` if the student received a free lunch in kindergarten, `non-free` otherwise (a measure of family income) |
| `schoolidk` | school identifier in kindergarten |

Documentation: <https://vincentarelbundock.github.io/Rdatasets/doc/AER/STAR.html>

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import norm

star = sm.datasets.get_rdataset("STAR", "AER").data   # downloads the data: needs internet
print(star.shape)
star[["stark", "readk", "mathk", "gender", "ethnicity", "lunchk", "schoolidk"]].head()

## Part 1 — Analyse the experiment (like Lab 2)

### 1.1 The sample

We compare **small** classes with **regular** classes (we leave `regular+aide` aside for now), and use the reading score `readk`.

The treatment variable is $D_i = 1$ if student $i$ was in a small class and $D_i = 0$ if in a regular class. The outcome is $Y_i$ = `readk`. We keep students with a recorded class type and reading score.

In [ ]:
kinder = star[star["stark"].isin(["small", "regular"])].dropna(subset=["readk"]).copy()
kinder["small"] = (kinder["stark"] == "small").astype(int)   # D_i: 1 = small class, 0 = regular class

small = kinder[kinder["small"] == 1]
regular = kinder[kinder["small"] == 0]
print("small:", len(small), "  regular:", len(regular))

### 1.2 Difference in means by hand

Let $n_1$ be the number of students in small classes and $n_0$ the number in regular classes. Compute

$$
\overline{Y}_1 = \frac{1}{n_1} \sum_{i \in \text{small}} Y_i, \qquad
\overline{Y}_0 = \frac{1}{n_0} \sum_{i \in \text{regular}} Y_i, \qquad
\hat{\tau} = \overline{Y}_1 - \overline{Y}_0 .
$$

Use `.sum()` and `len()`, as in Lab 2. Do **not** use `.mean()` here.

In [ ]:
# TODO: replace each None
n_1 = None      # number of students in small classes
n_0 = None      # number of students in regular classes
ybar_1 = None   # mean reading score in small classes, computed as sum / count
ybar_0 = None   # mean reading score in regular classes, computed as sum / count
tau_hat = None  # difference in means

In [ ]:
# Check (do not edit)
if tau_hat is None:
    print("not done yet")
else:
    reference = small["readk"].mean() - regular["readk"].mean()
    print("matches" if np.isclose(tau_hat, reference) else "does not match: check your sums and counts")

### 1.3 Standard error and 95% interval by hand

$$
\hat{S}_1^2 = \frac{1}{n_1 - 1} \sum_{i \in \text{small}} (Y_i - \overline{Y}_1)^2, \qquad
\hat{S}_0^2 = \frac{1}{n_0 - 1} \sum_{i \in \text{regular}} (Y_i - \overline{Y}_0)^2,
$$

$$
\text{SE} = \sqrt{\frac{\hat{S}_1^2}{n_1} + \frac{\hat{S}_0^2}{n_0}}, \qquad
\text{CI}_{95} = \left[\hat{\tau} - C_{0.975}\,\text{SE},\; \hat{\tau} + C_{0.975}\,\text{SE}\right],
$$

where $\hat{S}_1^2$ and $\hat{S}_0^2$ are the sample variances of the reading score in each group and $C_{0.975} \approx 1.96$ is the 97.5% quantile of the standard normal distribution (`norm.ppf(0.975)`).

In [ ]:
# TODO: replace each None (use the formulas above, not .var())
s2_1 = None    # sample variance in small classes
s2_0 = None    # sample variance in regular classes
se = None      # standard error of tau_hat
ci_low = None  # lower end of the 95% interval
ci_high = None # upper end of the 95% interval

In [ ]:
# Check (do not edit)
if se is None:
    print("not done yet")
else:
    reference = np.sqrt(small["readk"].var() / len(small) + regular["readk"].var() / len(regular))
    print("matches" if np.isclose(se, reference) else "does not match: check n - 1 and the square root")

### 1.4 The same numbers in one line

Write the `statsmodels` regression of `readk` on `small` with `cov_type="HC2"`, as in Lab 2 ("The same numbers in one line"), and print the coefficient on `small`, its standard error and the 95% interval. They should equal your numbers from 1.2 and 1.3.

In [ ]:
# TODO: replace None with the fitted regression
model = None

if model is not None:
    print("Difference in means:", model.params["small"].round(2))
    print("Standard error:     ", model.bse["small"].round(2))
    print("95% interval:       ", model.conf_int().loc["small"].round(2).tolist())

**Question 1.** In words a parent would understand: what is the effect of a small class on the kindergarten reading score, and how precise is it? To judge the size, compare it with the standard deviation of `readk` (`kinder["readk"].std()`).

*Your answer to Question 1:*

### 1.5 Balance check

Randomization should make the two groups similar **before** treatment. Build 0/1 indicators and compare the groups with standardized differences, as in Lab 2:

$$
\text{std. diff.} = \frac{\overline{X}_1 - \overline{X}_0}{\sqrt{(\hat{S}_{X,1}^2 + \hat{S}_{X,0}^2)/2}},
$$

where $\overline{X}_1, \overline{X}_0$ are the group means of a covariate $X$ and $\hat{S}_{X,1}^2, \hat{S}_{X,0}^2$ its sample variances in each group.

In [ ]:
kinder["female"] = (kinder["gender"] == "female").astype(float)
kinder["afam"] = (kinder["ethnicity"] == "afam").astype(float)
kinder["free_lunch"] = kinder["lunchk"].map({"free": 1.0, "non-free": 0.0})   # missing stays missing

small = kinder[kinder["small"] == 1]       # rebuild the groups so they include the new columns
regular = kinder[kinder["small"] == 0]
balance_vars = ["female", "afam", "free_lunch"]

# TODO: build a table with columns "small", "regular" and "std. diff." (one row per variable)
balance = None
balance

**Question 2.** Do the groups look balanced?

**Question 3.** `lunchk` was recorded during the kindergarten year, that is, *after* the class assignment. Can it still be used in a balance check? What would have to be true?

*Your answer to Questions 2 and 3:*

### 1.6 Optional: randomization happened within schools

In STAR, students were randomized **within each school**, and schools differ. Add school indicators to the regression with `+ C(schoolidk)` and compare the coefficient on `small` with 1.4.

**Optional question.** Why might the estimate change a little, and the standard error shrink, once we compare students only within the same school?

In [ ]:
# Optional: write the regression with school indicators here

## Part 2 — What if families had chosen? (like Lab 3)

In STAR, a lottery decided the class type. Suppose instead that small classes went to **families who asked for one**, and that families *not* on free lunch asked much more often.

We can build such a world from the real data. Each family "asks" with probability 0.8 if the child is not on free lunch and 0.2 if on free lunch. We then keep

- children in small classes whose families asked (they look like people who chose small classes), and
- children in regular classes whose families did not ask (they look like people who chose regular classes).

Because the lottery still decided who was in which class *within* each free-lunch group, we know the right answer: the experimental estimate.

In [ ]:
rng = np.random.default_rng(481)

both = kinder.dropna(subset=["free_lunch"]).copy()        # we need free-lunch status for everyone here
ask_prob = np.where(both["free_lunch"] == 1, 0.2, 0.8)    # chance that the family asks for a small class
both["asked"] = rng.binomial(1, ask_prob)

chosen = both[((both["small"] == 1) & (both["asked"] == 1)) | ((both["small"] == 0) & (both["asked"] == 0))]

experimental = both.loc[both["small"] == 1, "readk"].mean() - both.loc[both["small"] == 0, "readk"].mean()
print("Experimental difference (same students as below, before choosing):", round(experimental, 2))
print("Students kept:", len(chosen))

### 2.1 The naive comparison

Compute the difference in mean reading scores between small and regular classes **in `chosen`**.

In [ ]:
# TODO: replace None
naive = None
print("Naive difference:", naive)

**Question 4.** Is the naive difference larger or smaller than the experimental one? In LaLonde's comparison (Lab 3.3) the bias was negative. Which direction is it here, and why? (Hint: compare the share on free lunch in each group of `chosen`, and the mean reading score by free-lunch status.)

In [ ]:
# Use this cell for the hint (no TODO check here)

*Your answer to Question 4:*

### 2.2 Compare within groups

Now compute the difference in means **separately** for children on free lunch and children not on free lunch, inside `chosen`.

In [ ]:
# TODO: replace each None
gap_free = None       # small minus regular, among children on free lunch
gap_not_free = None   # small minus regular, among children not on free lunch
print("On free lunch:    ", gap_free)
print("Not on free lunch:", gap_not_free)

**Question 5.** How close are the within-group differences to the experimental difference? Each one uses fewer students than the naive comparison, so it is noisier: compute its standard error with the formula from 1.3 and check whether the experimental difference lies inside its 95% interval. Why does comparing within free-lunch groups work in this world? Name the assumption from the lecture that makes it work.

**Question 6.** Suppose the decision to ask also depended on how motivated the parents were, which is not in the data. Would comparing within free-lunch groups still work? What does this have to do with Lab 3.4?

**Question 7 (optional).** Change the seed in `default_rng(481)` and run Part 2 again. Which numbers move, and by how much?

In [ ]:
# Use this cell for Question 5 (no TODO check here)

*Your answer to Questions 5, 6 and 7:*

## References

- Krueger, A. B. (1999). Experimental estimates of education production functions. *Quarterly Journal of Economics*, 114(2), 497–532.
- Stock, J. H., and Watson, M. W. (2007). *Introduction to Econometrics*, 2nd ed. Addison Wesley. Data distributed in the R package AER; documentation: <https://vincentarelbundock.github.io/Rdatasets/doc/AER/STAR.html>
- Week 2 page: <https://yoonismo.github.io/causal-computing/week2.html>